# Un paso más con tu proyecto

**Sesión 5 · IA Intermedio · EstudIA 2026**

En la sesión 4 mediste tu modelo base contra una línea base. Hoy das un paso más, según tu proyecto:

| Camino | Para quién | Qué haces |
|---|---|---|
| **A1. Texto** | Tu proyecto clasifica textos | Miras cómo lee tu modelo tus textos (sus tokens), entrenas una bolsa de palabras con tus ejemplos y la comparas con lo que midió tu modelo base en la sesión 4 |
| **A2. Imagen** | Tu proyecto clasifica imágenes | Ajustas tu modelo base con tus fotos etiquetadas, con la receta de la sesión 4, y lo comparas con el modelo sin ajustar |
| **B. Práctica** | Todavía no tienes ejemplos | Haces A1 con los 30 mensajes ciudadanos de la sesión 4 |

Cuando termines, **entrega el notebook** desde la última celda. No importa si el modelo acierta.

**Antes de empezar.** *Archivo › Guardar una copia en Drive*. Para A1 y B no necesitas GPU. Para A2 sí: actívala
en *Entorno de ejecución › Cambiar tipo de entorno de ejecución › GPU T4*.

In [ ]:
# Preparación: librerías. Ejecútala una vez.
import os, re, time, zipfile, urllib.request
from collections import Counter
ARCHIVOS = os.environ.get("ESTUDIA_ARCHIVOS", "https://storage.googleapis.com/estud-ia-course-estudia-lab/s05")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from transformers import AutoTokenizer, pipeline

GPU = torch.cuda.is_available()
print("Listo.", f"GPU: {torch.cuda.get_device_name(0)}." if GPU else "Sin GPU: alcanza para A1 y B.")

In [ ]:
# @title Funciones auxiliares: descargar, números, tokens, evaluar, bolsa de palabras y vectores
def descargar(nombre, url=None):
    """Baja un archivo (de la carpeta pública del curso, o de `url`) solo la primera vez y devuelve su ruta."""
    ruta = os.path.join("archivos_curso", nombre)
    if not os.path.exists(ruta):
        os.makedirs(os.path.dirname(ruta), exist_ok=True)
        urllib.request.urlretrieve(url or f"{ARCHIVOS}/{nombre}", ruta)
    return ruta

def miles(n):
    """12345 -> "12.345", como escribimos los números en español."""
    return f"{n:,}".replace(",", ".")

def pct(x, decimales=1):
    """0.958 -> "95,8 %"."""
    return f"{100 * x:.{decimales}f} %".replace(".", ",")

def tokens(texto):
    """Parte un texto en palabras y signos, en minúsculas: "No me gustó." -> ["no", "me", "gustó", "."]."""
    return re.findall(r"\w+|[^\w\s]", str(texto).lower())

def evaluar(verdad, prediccion, titulo="Tu modelo"):
    """Compara las predicciones con las etiquetas reales y con la línea base de la clase más frecuente."""
    verdad, prediccion = [str(v) for v in verdad], [str(p) for p in prediccion]
    clases = sorted(set(verdad))
    frecuente = max(clases, key=verdad.count)
    def balanceada(p):
        return float(np.mean([np.mean([pi == c for vi, pi in zip(verdad, p) if vi == c]) for c in clases]))
    print(f"{titulo} · {len(verdad)} ejemplos, {len(clases)} clases")
    print(f"  Exactitud:            {pct(np.mean([v == p for v, p in zip(verdad, prediccion)]))}   "
          f"(línea base “siempre {frecuente}”: {pct(np.mean([v == frecuente for v in verdad]))})")
    print(f"  Exactitud balanceada: {pct(balanceada(prediccion))}   (línea base: {pct(balanceada([frecuente] * len(verdad)))})")
    todas = clases + sorted(set(prediccion) - set(clases))
    fig, eje = plt.subplots(figsize=(1.1 * len(todas) + 2.5, 0.9 * len(clases) + 1.8))
    ConfusionMatrixDisplay(confusion_matrix(verdad, prediccion, labels=todas), display_labels=todas).plot(
        ax=eje, cmap="Purples", colorbar=False, xticks_rotation=45)
    eje.set_xlabel("Lo que respondió"); eje.set_ylabel("La etiqueta real"); eje.set_title(titulo)
    eje.set_ylim(len(clases) - 0.5, -0.5)
    plt.tight_layout(); plt.show()

def _pliegues(etiquetas):
    n = int(min(5, np.bincount(np.unique(etiquetas, return_inverse=True)[1]).min()))
    if n < 2:
        print("Necesitas al menos 2 ejemplos de cada etiqueta.")
    return n

def bolsa_con_validacion_cruzada(textos, etiquetas):
    """Bolsa de palabras + regresión logística, medida con validación cruzada: cada ejemplo se predice con un modelo
    que no lo vio. Con pocos ejemplos es la forma honesta de medir."""
    etiquetas = np.array([str(e) for e in etiquetas])
    n = _pliegues(etiquetas)
    if n < 2:
        return None
    modelo = make_pipeline(TfidfVectorizer(tokenizer=tokens, lowercase=False, token_pattern=None, ngram_range=(1, 2)),
                           LogisticRegression(max_iter=3000, C=10, class_weight="balanced"))
    return cross_val_predict(modelo, list(textos), etiquetas, cv=StratifiedKFold(n, shuffle=True, random_state=0))

PALABRAS_VACIAS = set("""el la los las lo un una unos unas de del al a en y o que se su sus por con para
    no me mi mis te tu es son fue hay ya le les nos este esta esto ese esa como pero muy más""".split())

def vector_de_texto(texto):
    """El promedio de los vectores de fastText de las palabras del texto, sin las palabras vacías."""
    vs = [v for w in re.findall(r"\w+", str(texto)) if w.lower() not in PALABRAS_VACIAS
          for v in [vector_de(w)] if v is not None]
    return np.mean(vs, axis=0) if vs else np.zeros(300, dtype=np.float32)

def vectores_con_validacion_cruzada(textos, etiquetas):
    """Cada texto se vuelve el promedio de los vectores de sus palabras; encima, una regresión logística. Misma
    validación cruzada que la bolsa de palabras."""
    etiquetas = np.array([str(e) for e in etiquetas])
    n = _pliegues(etiquetas)
    if n < 2:
        return None
    X = np.stack([vector_de_texto(t) for t in textos])
    return cross_val_predict(LogisticRegression(max_iter=3000, C=10), X, etiquetas,
                             cv=StratifiedKFold(n, shuffle=True, random_state=0))

def mirar_tokens(tokenizador, textos, cuantos=5):
    """Cómo parte el tokenizador de tu modelo cada texto, y cuántas palabras rompe en pedazos."""
    rotas, total = Counter(), 0
    for t in textos:
        piezas = tokenizador.tokenize(str(t))
        total += len(piezas)
        for palabra in re.findall(r"\w+", str(t)):
            if len(tokenizador.tokenize(palabra)) > 1:
                rotas[palabra.lower()] += 1
    for t in list(textos)[:cuantos]:
        print("•", str(t)[:80]); print("  ", tokenizador.tokenize(str(t))[:30])
    largo = getattr(tokenizador, "model_max_length", None)
    print(f"\nTokens por texto, en promedio: {total / max(1, len(textos)):.1f}".replace(".", ","),
          f"· el modelo lee hasta {largo} tokens." if largo and largo < 100000 else "")
    print("Palabras que el tokenizador parte en pedazos (las más frecuentes):", ", ".join(w for w, _ in rotas.most_common(15)))

_VECTORES = None

def vectores():
    """Las 100.000 palabras más frecuentes del español con su vector de 300 números (fastText). Se baja una vez."""
    global _VECTORES
    if _VECTORES is None:
        datos = np.load(descargar("vectores_es.npz"))
        matriz = datos["vectores"].astype(np.float32)
        matriz /= np.linalg.norm(matriz, axis=1, keepdims=True)
        palabras = datos["palabras"].tolist()
        _VECTORES = (palabras, {p: i for i, p in enumerate(palabras)}, matriz)
    return _VECTORES

def vector_de(palabra):
    """El vector de una palabra, o None si no está entre las 100.000. Prueba también con mayúscula inicial."""
    palabras, indice, matriz = vectores()
    for forma in (palabra, palabra.lower(), palabra.capitalize()):
        if forma in indice:
            return matriz[indice[forma]]
    return None

def vecinos(palabra, cuantos=6):
    """Las palabras con el vector más parecido (similitud del coseno)."""
    palabras, indice, matriz = vectores()
    v = vector_de(palabra)
    if v is None:
        return f"“{palabra}” no está entre las 100.000 palabras."
    orden = np.argsort(-(matriz @ v))
    return [palabras[i] for i in orden[:cuantos + 3] if palabras[i].lower() != palabra.lower()][:cuantos]

def analogia(a, b, c, cuantas=4):
    """b - a + c: “París es a Francia lo que ___ es a Colombia” es analogia("Francia", "París", "Colombia")."""
    palabras, indice, matriz = vectores()
    va, vb, vc = vector_de(a), vector_de(b), vector_de(c)
    q = vb - va + vc
    orden = np.argsort(-(matriz @ (q / np.linalg.norm(q))))
    fuera = {a.lower(), b.lower(), c.lower()}
    return [palabras[i] for i in orden[:20] if palabras[i].lower() not in fuera][:cuantas]

---
## A1. Texto

### Tus ejemplos y tu modelo

Los mismos de la sesión 4: escríbelos en `EJEMPLOS` o sube `ejemplos.csv` (columnas `entrada` y `etiqueta`). Escribe
en `MODELO` tu modelo base. Sin datos personales.

In [ ]:
EJEMPLOS = [
    ("Escribe aquí un ejemplo real de tu proyecto.", "etiqueta_1"),
    ("Y otro, con otra etiqueta.", "etiqueta_2"),
]
MODELO = "dueño/nombre-del-modelo"   # tu modelo base, por ejemplo: "cardiffnlp/twitter-xlm-roberta-base-sentiment"

if os.path.exists("ejemplos.csv"):
    tabla = pd.read_csv("ejemplos.csv")
    EJEMPLOS = list(zip(tabla["entrada"].astype(str), tabla["etiqueta"].astype(str)))
mis_textos = not EJEMPLOS[0][0].startswith("Escribe aquí")
if mis_textos:
    textos, etiquetas = [e for e, _ in EJEMPLOS], [str(et) for _, et in EJEMPLOS]
    print(f"{len(textos)} ejemplos. Etiquetas:", pd.Series(etiquetas).value_counts().to_dict())
else:
    print("Camino A1: escribe tus ejemplos en EJEMPLOS (o sube ejemplos.csv) y vuelve a ejecutar.")

### 1. Cómo lee tu modelo tus textos

Los modelos de Hugging Face no usan palabras enteras: su tokenizador parte las palabras raras en pedazos. Mira cómo
quedan las palabras de tu proyecto: jerga, nombres de lugares, errores de ortografía.

In [ ]:
if mis_textos and not MODELO.startswith("dueño/"):
    tokenizador = AutoTokenizer.from_pretrained(MODELO)
    mirar_tokens(tokenizador, textos)
elif mis_textos:
    print("Escribe tu modelo en MODELO y vuelve a ejecutar.")

### 2. Dos clasificadores entrenados con tus ejemplos

Con pocos ejemplos no conviene apartar una prueba fija: usamos **validación cruzada**. Partimos los ejemplos en
grupos; cada grupo se predice con un modelo entrenado con los demás. Así cada ejemplo se predice sin haberlo visto.

Comparamos dos formas de convertir tus textos en números: una **bolsa de palabras**, y el **promedio de los
vectores de fastText** de sus palabras (sin las palabras vacías como “el”, “de” o “que”).

In [ ]:
if mis_textos:
    pred = bolsa_con_validacion_cruzada(textos, etiquetas)
    if pred is not None:
        evaluar(etiquetas, pred, "Bolsa de palabras con tus ejemplos")
        evaluar(etiquetas, vectores_con_validacion_cruzada(textos, etiquetas), "Promedio de vectores con tus ejemplos")

Escribe aquí lo que midió tu modelo base en la sesión 4 (exactitud y exactitud balanceada) para compararlo:

**Mi modelo base en la sesión 4** (doble clic aquí):

### 3. Las palabras de tu proyecto, como vectores

Escribe algunas palabras importantes de tu proyecto y mira sus vecinas en los vectores de fastText.

In [ ]:
MIS_PALABRAS = ["queja", "servicio", "factura"]   # cámbialas por palabras de tu proyecto
for palabra in MIS_PALABRAS:
    print(f"{palabra:14s} → {vecinos(palabra)}")

**Lo que observé** (doble clic aquí y responde en dos o tres líneas):

- ¿Qué palabras de tu proyecto rompe el tokenizador en pedazos? ¿Son importantes para la etiqueta?
- ¿Cuál de los dos clasificadores le gana al otro? ¿Alguno le gana a tu modelo base de la sesión 4? ¿Qué te dice eso?
- ¿Las vecinas de tus palabras tienen sentido en tu contexto, o el vector aprendió otro significado?

---
## A2. Imagen

Sube un archivo `mis_fotos.zip` con una carpeta por etiqueta (por ejemplo, `mis_fotos/sana/…jpg` y
`mis_fotos/enferma/…jpg`), sin personas reconocibles. Escribe tu modelo base en `MODELO_IMAGEN`. Apartamos el 30 % de
las fotos para medir; con pocas fotos, el resultado es ruidoso: tómalo como una primera señal.

In [ ]:
MODELO_IMAGEN = "dueño/nombre-del-modelo"   # por ejemplo: "facebook/convnext-tiny-224"
CONGELAR = False
EPOCAS = 10
TASA = 1e-4

mis_fotos = os.path.exists("mis_fotos.zip") and not MODELO_IMAGEN.startswith("dueño/")
if mis_fotos:
    with zipfile.ZipFile("mis_fotos.zip") as z:
        z.extractall(".")
    from PIL import Image
    rutas = sorted(os.path.join(r, f) for r, _, fs in os.walk("mis_fotos") for f in fs
                   if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp")))
    etiquetas_img = [os.path.basename(os.path.dirname(r)) for r in rutas]
    CLASES_IMG = sorted(set(etiquetas_img))
    fotos = [Image.open(r).convert("RGB").resize((224, 224)) for r in rutas]
    i_ent, i_prueba = train_test_split(np.arange(len(fotos)), test_size=0.3, random_state=0,
                                       stratify=etiquetas_img)
    print(f"{len(fotos)} fotos, {len(CLASES_IMG)} etiquetas: {Counter(etiquetas_img)}. "
          f"Entrenamiento {len(i_ent)} · prueba {len(i_prueba)}.")
else:
    print("Camino A2: sube mis_fotos.zip, escribe MODELO_IMAGEN y vuelve a ejecutar.")

In [ ]:
if mis_fotos:
    from transformers import AutoImageProcessor, AutoModelForImageClassification, Trainer, TrainingArguments
    procesador = AutoImageProcessor.from_pretrained(MODELO_IMAGEN)
    class Fotos(torch.utils.data.Dataset):
        def __init__(self, idx): self.idx = idx
        def __len__(self): return len(self.idx)
        def __getitem__(self, k):
            i = self.idx[k]
            x = procesador(images=fotos[i], return_tensors="pt")["pixel_values"][0]
            return {"pixel_values": x, "labels": CLASES_IMG.index(etiquetas_img[i])}
    modelo_img = AutoModelForImageClassification.from_pretrained(
        MODELO_IMAGEN, id2label=dict(enumerate(CLASES_IMG)), label2id={c: i for i, c in enumerate(CLASES_IMG)},
        ignore_mismatched_sizes=True)
    if CONGELAR:
        for nombre, p in modelo_img.named_parameters():
            p.requires_grad = "classifier" in nombre
    args = TrainingArguments(output_dir="ajuste", num_train_epochs=EPOCAS, learning_rate=TASA,
                             per_device_train_batch_size=8, warmup_steps=0.1, save_strategy="no",
                             logging_strategy="epoch", fp16=GPU, report_to="none", seed=0)
    trainer = Trainer(model=modelo_img, args=args, train_dataset=Fotos(i_ent))
    inicio = time.time(); trainer.train(); print(f"Ajuste en {time.time() - inicio:.0f} s.")
    pred = trainer.predict(Fotos(i_prueba)).predictions.argmax(-1)
    evaluar([etiquetas_img[i] for i in i_prueba], [CLASES_IMG[k] for k in pred], "Tu modelo ajustado")

**Lo que observé** (doble clic aquí y responde): ¿el modelo ajustado le gana a lo que midió tu modelo base en la
sesión 4? ¿Qué etiqueta confunde? ¿Cuántas fotos más crees que necesitarías?

---
## B. Práctica: los mensajes ciudadanos

Los 30 mensajes de la sesión 4, cinco por tema. Allí, el modelo zero-shot acertó el 63 % con etiquetas
descriptivas. ¿Qué logran una bolsa de palabras y un promedio de vectores entrenados con estos mismos 30 mensajes?
La bolsa de palabras ya está; la medición con vectores la completas tú (si te atascas, abre la **pista** que está
debajo de la celda).

In [ ]:
MENSAJES = [
    ("El semáforo de la 80 con la 33 lleva tres días apagado y se forman unos trancones terribles.", "movilidad"),
    ("El metro iba tan lleno esta mañana que no pude subirme en tres trenes.", "movilidad"),
    ("Los buses de la ruta del barrio dejaron de pasar después de las ocho de la noche.", "movilidad"),
    ("Hay un hueco enorme en la vía principal y ya se han caído dos motociclistas.", "movilidad"),
    ("No hay señalización en la nueva ciclorruta y los carros se parquean encima.", "movilidad"),
    ("Anoche robaron dos celulares en el paradero del parque y no había ningún policía.", "seguridad"),
    ("Pusieron cámaras en la ciclorruta, pero siguen robando bicicletas.", "seguridad"),
    ("En la esquina de la cancha venden droga a plena luz del día.", "seguridad"),
    ("Las luces del puente peatonal están dañadas y da miedo pasar de noche.", "seguridad"),
    ("Unos muchachos están cobrando vacuna a los tenderos de la cuadra.", "seguridad"),
    ("En el centro de salud del barrio no hay citas para medicina general hasta el próximo mes.", "salud"),
    ("Llevo tres horas esperando en urgencias con mi mamá y nadie nos atiende.", "salud"),
    ("La farmacia de la EPS no tiene la insulina desde hace dos semanas.", "salud"),
    ("Hay muchos casos de dengue en el sector y nadie ha venido a fumigar.", "salud"),
    ("Necesitamos una jornada de vacunación para los niños de la vereda.", "salud"),
    ("La escuela de la vereda no tiene profesor de matemáticas desde marzo.", "educación"),
    ("Al colegio no ha llegado el refrigerio de los estudiantes en todo el mes.", "educación"),
    ("Los computadores de la sala de informática del colegio llevan un año dañados.", "educación"),
    ("No hay cupos en el preescolar del barrio y tengo que trabajar.", "educación"),
    ("El techo del salón se cae a pedazos cuando llueve y los niños reciben clase así.", "educación"),
    ("Hace una semana se fue el agua en toda la cuadra y nadie da razón.", "servicios públicos"),
    ("La factura de la luz llegó el doble de lo normal sin ninguna explicación.", "servicios públicos"),
    ("Hay una fuga de gas en el contador del edificio y huele muy fuerte.", "servicios públicos"),
    ("El alcantarillado se rebosa cada vez que llueve y el agua entra a las casas.", "servicios públicos"),
    ("Se cayó un cable de energía en el andén y está echando chispas.", "servicios públicos"),
    ("Están quemando basura al lado de la quebrada y el humo llega a todas las casas.", "medio ambiente"),
    ("Talaron los árboles del parque sin ningún permiso.", "medio ambiente"),
    ("Una fábrica está botando aguas negras al río.", "medio ambiente"),
    ("La calidad del aire está pésima y los niños tienen tos todo el día.", "medio ambiente"),
    ("Dejan escombros y colchones viejos en la orilla de la quebrada.", "medio ambiente"),
]
textos_b, temas_b = [m for m, _ in MENSAJES], [t for _, t in MENSAJES]
pred_bolsa = bolsa_con_validacion_cruzada(textos_b, temas_b)
evaluar(temas_b, pred_bolsa, "Bolsa de palabras, 30 mensajes")

# Completa: la misma medición, pero con el promedio de vectores (la función vectores_con_validacion_cruzada).
pred_vectores = ...

if pred_vectores is ...:
    print("Completa la línea de pred_vectores y vuelve a ejecutar. Si te atascas, abre la pista de abajo.")
else:
    evaluar(temas_b, pred_vectores, "Promedio de vectores, 30 mensajes")

tokenizador_b = AutoTokenizer.from_pretrained("MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli")
mirar_tokens(tokenizador_b, textos_b, cuantos=3)
for palabra in ["trancones", "vereda", "EPS", "alcantarillado"]:
    print(f"{palabra:15s} → {vecinos(palabra)}")

<details><summary>Pista</summary>

Mira la línea de `pred_bolsa`: recibe los textos y los temas, y devuelve una predicción por mensaje. La de vectores
recibe lo mismo; solo cambia la función, y su nombre está en el comentario de la celda:

```python
pred_vectores = …(textos_b, temas_b)
```
</details>

**Lo que observé** (doble clic aquí y responde): ¿cuál le gana a cuál: la bolsa de palabras, el promedio de
vectores o el zero-shot de la sesión 4? Con solo cinco ejemplos por tema, ¿por qué los vectores ayudan tanto?

---
## Entrega

Ejecuta las dos celdas siguientes. La primera solo prepara la función; la segunda lee este notebook con sus
resultados y lo envía a la plataforma. Te pide tu clave la primera vez.

Si algo falla, descarga el notebook (*Archivo › Descargar › Descargar .ipynb*) y súbelo en la plataforma, en la
actividad “Entrega: un paso más con tu proyecto”.

In [ ]:
# @title Función para entregar el notebook (no hace falta cambiarla)
import getpass, json, urllib.error

PLATAFORMA = os.environ.get("ESTUDIA_PLATAFORMA", "https://estud-ia.eslopezag.com")
_CLAVE = os.environ.get("ESTUDIA_CLAVE", "")
ACTIVIDAD = "“Entrega: un paso más con tu proyecto”"

def _este_notebook():
    """El notebook tal como se ve en Colab, con sus resultados. Fuera de Colab, el archivo de ESTUDIA_NOTEBOOK."""
    if os.environ.get("ESTUDIA_NOTEBOOK"):
        with open(os.environ["ESTUDIA_NOTEBOOK"], encoding="utf-8") as f:
            return json.load(f)
    try:
        from google.colab import _message
        return _message.blocking_request("get_ipynb", timeout_sec=60)["ipynb"]
    except Exception:
        return None

def entregar():
    """Envía este notebook, con lo que ejecutaste, a la plataforma del curso."""
    global _CLAVE
    cuaderno = _este_notebook()
    if not cuaderno:
        print("No pude leer el notebook desde aquí. Descárgalo (Archivo › Descargar › Descargar .ipynb) y súbelo "
              f"en la plataforma, en la actividad {ACTIVIDAD}.")
        return
    if not _CLAVE:
        _CLAVE = getpass.getpass(f"Pega tu clave (plataforma › {ACTIVIDAD} › Copiar) y oprime Enter: ").strip()
    req = urllib.request.Request(f"{PLATAFORMA}/api/entregas/s05", data=json.dumps({"notebook": cuaderno}).encode(),
                                 method="POST", headers={"Content-Type": "application/json",
                                                         "Authorization": f"Bearer {_CLAVE}",
                                                         "User-Agent": "EstudIA-notebook/1.0"})
    try:
        with urllib.request.urlopen(req, timeout=120) as r:
            res = json.loads(r.read())
    except urllib.error.HTTPError as e:
        res = json.loads(e.read() or b"{}")
        if e.code == 401:
            _CLAVE = ""
        print("No se pudo entregar:", res.get("error", f"error {e.code}"))
        print(f"Si no se arregla, descarga el notebook y súbelo en la plataforma, en la actividad {ACTIVIDAD}.")
        return
    except urllib.error.URLError:
        print("No hay conexión con la plataforma. Revisa tu internet o pregúntale al profesor si está encendida.")
        return
    modelos = ", ".join(res["modelos"]) or "ninguno"
    print(f"Entrega recibida. Celdas de código ejecutadas: {res['ejecutadas']} de {res['celdas']}. "
          f"Modelos de Hugging Face que encontré: {modelos}.")
    for aviso in res.get("avisos", []):
        print("Ojo:", aviso)

In [ ]:
entregar()